# Fantasy value backtest: does last season predict this season?

The board blends seasons of category z-scores. The question behind the blend is how well a
ranking you could make before a season, last season's full-season z-score, predicts the value a
player goes on to produce. This notebook measures it for every pair of consecutive seasons in each
league (players with at least ten games in the later season), and names the biggest misses.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe, leagues, save_findings

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
LEAGUES = leagues(con)
LABEL = {'wnba': 'WNBA', 'nba': 'NBA'}
COLOR = {'wnba': '#b23a0a', 'nba': '#1a7f4b'}
findings = {}
print(describe(con), 'Leagues:', ', '.join(LABEL[lg] for lg in LEAGUES))

In [ ]:
z = con.sql('''
    select league, season, player_id, games, z_total
    from mart_category_zscores
    where window_name = 'season' and in_pool
''').df()
pairs, latest_pairs = [], {}
for lg in LEAGUES:
    part = z[z['league'] == lg]
    seasons = sorted(part['season'].unique())
    for prev, cur in zip(seasons, seasons[1:], strict=False):
        a = part[part['season'] == prev][['player_id', 'z_total']].rename(columns={'z_total': 'prior'})
        b = part[(part['season'] == cur) & (part['games'] >= 10)][['player_id', 'z_total', 'games']].rename(columns={'z_total': 'actual'})
        both = a.merge(b, on='player_id')
        if len(both) >= 30:
            top = both.nlargest(20, 'prior')
            pairs.append({'league': lg, 'prior_season': prev, 'season': cur, 'players': len(both),
                          'spearman': round(both[['prior', 'actual']].corr(method='spearman').iloc[0, 1], 3),
                          'top20_still_top20': int(top['player_id'].isin(both.nlargest(20, 'actual')['player_id']).sum())})
            latest_pairs[lg] = (prev, cur, both)
backtest = pd.DataFrame(pairs, columns=['league', 'prior_season', 'season', 'players', 'spearman', 'top20_still_top20'])
backtest

In [ ]:
if len(backtest):
    fig, ax = plt.subplots(figsize=(8, 4))
    for lg in LEAGUES:
        part = backtest[backtest['league'] == lg]
        ax.plot(part['season'], part['spearman'], marker='o', label=LABEL[lg], color=COLOR[lg])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Spearman rank correlation')
    ax.legend()
    ax.set_title("How well last season's z-score rank predicted the next")
    plt.tight_layout()
    plt.show()

## The biggest misses

In the most recent pair of seasons per league: the risers a preseason rank would have missed and
the fallers it would have overrated.

In [ ]:
names = con.sql("select league, stats_player_id as player_id, full_name from dim_player where is_current").df()
moves = {}
for lg, (prev, cur, both) in latest_pairs.items():
    m = both.assign(change=both['actual'] - both['prior']).merge(
        names[names['league'] == lg][['player_id', 'full_name']], on='player_id', how='left')
    moves[lg] = m
    print(f'{LABEL[lg]}: {prev} to {cur}')
    display(m.nlargest(8, 'change')[['full_name', 'games', 'prior', 'actual', 'change']].round(2))
    display(m.nsmallest(8, 'change')[['full_name', 'games', 'prior', 'actual', 'change']].round(2))

In [ ]:
for lg in LEAGUES:
    part = backtest[backtest['league'] == lg]
    if not len(part):
        continue
    prev, cur, _ = latest_pairs[lg]
    m = moves[lg]
    risers = m.nlargest(5, 'change')
    findings[lg] = {
        'headline': f"Last season's rank predicts the next with a rank correlation of {part['spearman'].mean():.2f} on average; {part.iloc[-1]['top20_still_top20']} of the top 20 in {prev} stayed top 20 in {cur}.",
        'facts': [
            {'label': 'Season pairs', 'value': str(len(part))},
            {'label': 'Rank correlation, latest pair', 'value': f"{part.iloc[-1]['spearman']:.2f} ({prev} to {cur})"},
            {'label': 'Best and worst pair', 'value': f"{part['spearman'].max():.2f} / {part['spearman'].min():.2f}"},
            {'label': 'Biggest riser', 'value': f"{risers.iloc[0]['full_name']}: {risers.iloc[0]['change']:+.2f}" if len(risers) else 'n/a'},
        ],
        'table': {
            'columns': ['Riser', f'{prev} value', f'{cur} value', 'Change'],
            'rows': [[r['full_name'], round(r['prior'], 2), round(r['actual'], 2), round(r['change'], 2)] for _, r in risers.iterrows()],
        },
    }
save_findings('03_fantasy_value_backtest', {'title': 'Fantasy value backtest', 'leagues': findings})
findings